# Can an LLM improve movie recommendation rankings?

Compare GPT-6 Luna with EASE and the IMDb credits hybrid on **100 seeded validation users**.
The LLM ranks **100 unseen movies retrieved by EASE** per user. All models share the same
shortlist for the controlled comparison; separate full-catalog baselines show end-to-end performance.

Run notebooks **01, 04, 09, and 10** first. The runner reconstructs and verifies notebook 10's
control, and uses EASE regularization **50**, fixed from notebook 08's previous selection.
No new baseline or prompt tuning is performed in this pilot.

- Only the last 30 **training** ratings and movie metadata enter prompts. Histories include low
  ratings too. Candidates exclude **all** training-rated movies and are shuffled with a per-user seed.
- Validation positives select neither candidates nor prompt content. They are never forcibly inserted
  into a shortlist, and positives outside the shortlist remain in Recall/NDCG denominators.
- Return exactly ten unique catalog IDs through a strict JSON schema. Invalid/incomplete responses
  receive an empty recommendation list (zero hits); users are never dropped. A separately labeled
  EASE fallback model shows the behavior of a deployable fallback policy.
- Record shortlist coverage, complete per-user results, paired bootstrap intervals, returned model
  versions, token usage, latency, and persistent response hashes.
- This remains **exploratory validation**. Current movie metadata and possible pretraining exposure
  to public MovieLens/Reddit content limit historical and generalization claims. Test feedback is not read.

## Access and budget

Use `OPENAI_API_KEY` in your environment, or a local `.env` file ignored by Git. Never paste keys
into notebook source or output. No extra packages are required beyond this project's environment.

**Default: no API calls.** Setting `RUN_LIVE = True` below enables requests. The runner caps
cumulative conservative reservations at **$1**, limits output to 512 tokens, and uses no tools,
reasoning tokens, automatic retries, or parallel requests. Successful and failed calls are cached.
An interrupted/uncertain call keeps its reservation; reruns refuse to repeat an uncached reservation.

Reservations use UTF-8 request bytes plus protocol allowance as a conservative input-token bound,
the published input/cache-write rates, and maximum output tokens. This is a client-side safeguard
at the recorded prices, **not a provider-enforced account billing limit**. Other API usage is outside
this ledger. Check provider pricing before a future refresh; do not delete the ledger to rerun calls.
Keep Auto Recharge off if you intend to use only your included API credits.

Sources: [model and pricing](https://developers.openai.com/api/docs/models/gpt-6-luna),
[structured outputs](https://developers.openai.com/api/docs/guides/structured-outputs),
[LLM ranking research](https://arxiv.org/abs/2305.08845).

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

project_root = next(path for path in (Path.cwd(), *Path.cwd().parents)
                    if (path / "scripts/llm_recommendation_experiment.py").is_file())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from scripts.llm_recommendation_experiment import run_experiment, digest

RUN_LIVE = False  # True enables up to $1 of cumulative reserved API usage.
RETRY_AUTH_FAILURES = False  # True only after replacing a key rejected with HTTP 401.
summary, report = run_experiment(
    project_root, live=RUN_LIVE, retry_auth_failures=RETRY_AUTH_FAILURES
)
output_dir = project_root / "data/processed/per_user_chronological_60_20_20_v1/notebook11_llm_reranking"
display(summary.round(6))

Prepared 100 users; maximum planned reservation $0.6631; no API calls.
                         model  users  mean_recall_at_10  mean_ndcg_at_10
             EASE full catalog    100           0.143138         0.149123
         EASE shared shortlist    100           0.143138         0.149123
      IMDb hybrid full catalog    100           0.157483         0.150806
  IMDb hybrid shared shortlist    100           0.160159         0.152936
          LLM shared shortlist    100           0.076267         0.086667
LLM with EASE failure fallback    100           0.084727         0.095838
{
  "complete": true,
  "planned_users": 100,
  "attempted_users": 100,
  "valid_users": 92,
  "estimated_token_cost_usd": 0.1205244,
  "cumulative_reserved_usd": 0.66944025,
  "cost_note": "Standard token estimate; cache adjustments ignored; unresolved calls remain reserved.",
  "plan_sha256": "c7e85e7199284996889624f1891e0e794576e3d7faf0aa55f9c105c2327355c0",
  "test_feedback_read": false,
  "files": {
   

,model,users,mean_recall_at_10,mean_ndcg_at_10
0,EASE full catalog,100,0.143138,0.149123
1,EASE shared shortlist,100,0.143138,0.149123
2,IMDb hybrid full catalog,100,0.157483,0.150806
3,IMDb hybrid shared shortlist,100,0.160159,0.152936
4,LLM shared shortlist,100,0.076267,0.086667
5,LLM with EASE failure fallback,100,0.084727,0.095838


## Inspect candidate coverage and uncertainty

`shortlist_recall` is the fraction of all validation likes retrieved in the 100-movie shortlist.
`maximum_recall_at_10` is the best Recall@10 any ranker could attain using that shortlist.
These are diagnostics, not ranking scores. A strong ranker cannot retrieve an omitted movie.

The paired intervals bootstrap **users**, keeping each LLM result paired with its baseline result.
They quantify uncertainty for this fixed pilot; they do not correct repeated validation tuning,
pretraining leakage, or prompt-order variance. The one fixed shuffled candidate order is a pilot
choice; evaluating additional orders would require additional budgeted requests.

In [2]:
coverage = pd.read_csv(output_dir / "shortlist_coverage.csv")
display(coverage[["shortlist_recall", "maximum_recall_at_10"]].describe().round(4))
print("Attempted users:", report["attempted_users"], "/", report["planned_users"])
print("Valid LLM responses:", report["valid_users"])
print("Estimated standard token cost: $", round(report["estimated_token_cost_usd"], 6))
print("Cumulative conservative reservation: $", round(report["cumulative_reserved_usd"], 6))
if report["complete"]:
    paired = pd.read_csv(output_dir / "paired_bootstrap.csv")
    usage = pd.read_csv(output_dir / "api_usage.csv")
    display(paired.round(6))
    display(usage.groupby("status").agg(users=("userId", "size"),
                                       mean_latency_seconds=("elapsed_seconds", "mean")))
else:
    print("Live pilot is incomplete. The summary contains baselines only; no partial LLM scores are reported.")

Attempted users: 100 / 100
Valid LLM responses: 92
Estimated standard token cost: $ 0.120524
Cumulative conservative reservation: $ 0.66944


,shortlist_recall,maximum_recall_at_10
count,100.0000,100.0000
mean,0.4132,0.4044
std,0.3132,0.3158
min,0.0000,0.0000
25%,0.1954,0.1780
50%,0.3333,0.3333
75%,0.5877,0.5877
max,1.0000,1.0000


,baseline,metric,mean_change,ci95_low,ci95_high,split_fingerprint,evaluation_split
0,EASE full catalog,recall_at_10,-0.066871,-0.108055,-0.028557,8a8613e8a10e363629f76f9aacaaf1952e15fcf0b3b002...,validation
1,EASE full catalog,ndcg_at_10,-0.062456,-0.102850,-0.024038,8a8613e8a10e363629f76f9aacaaf1952e15fcf0b3b002...,validation
2,EASE shared shortlist,recall_at_10,-0.066871,-0.108055,-0.028557,8a8613e8a10e363629f76f9aacaaf1952e15fcf0b3b002...,validation
3,EASE shared shortlist,ndcg_at_10,-0.062456,-0.102850,-0.024038,8a8613e8a10e363629f76f9aacaaf1952e15fcf0b3b002...,validation
4,IMDb hybrid full catalog,recall_at_10,-0.081216,-0.129054,-0.039194,8a8613e8a10e363629f76f9aacaaf1952e15fcf0b3b002...,validation
5,IMDb hybrid full catalog,ndcg_at_10,-0.064140,-0.101573,-0.027867,8a8613e8a10e363629f76f9aacaaf1952e15fcf0b3b002...,validation
6,IMDb hybrid shared shortlist,recall_at_10,-0.083892,-0.131270,-0.042949,8a8613e8a10e363629f76f9aacaaf1952e15fcf0b3b002...,validation
7,IMDb hybrid shared shortlist,ndcg_at_10,-0.066269,-0.103061,-0.030662,8a8613e8a10e363629f76f9aacaaf1952e15fcf0b3b002...,validation


,users,mean_latency_seconds
status,,
incomplete_or_refused,2,4.731253
invalid,6,2.893093
ok,92,1.455687


## Read example recommendations

The two lowest-ID sampled users are chosen without inspecting gains. Show the actual cached LLM
IDs with metadata supplied in the prompt, alongside EASE's top ten candidates. Output contains
no API key, and rerunning this inspection makes no network requests.

In [3]:
requests = json.loads((output_dir / "requests.json").read_text(encoding="utf-8"))
example_rows = []
for entry in requests[:2]:
    cache_path = output_dir / "responses" / (digest(entry["request"]) + ".json")
    if not cache_path.is_file():
        continue
    cached = json.loads(cache_path.read_text(encoding="utf-8"))
    candidate_data = json.loads(entry["request"]["input"])["candidate_movies"]
    lookup = {movie["movieId"]: movie for movie in candidate_data}
    for model, ids in [("EASE", entry["fallback_ids"]), ("LLM", cached["movie_ids"])]:
        for rank, item in enumerate(ids, 1):
            example_rows.append({"userId": entry["userId"], "model": model, "rank": rank,
                                 "movieId": item, "title": lookup[item]["title"],
                                 "is_validation_like": item in entry["positives"]})
display(pd.DataFrame(example_rows))

,userId,model,rank,movieId,title,is_validation_like
0,25,EASE,1,318,"Shawshank Redemption, The (1994)",False
1,25,EASE,2,1196,Star Wars: Episode V - The Empire Strikes Back...,False
2,25,EASE,3,58559,"Dark Knight, The (2008)",False
3,25,EASE,4,2762,"Sixth Sense, The (1999)",False
4,25,EASE,5,2028,Saving Private Ryan (1998),True
5,25,EASE,6,1210,Star Wars: Episode VI - Return of the Jedi (1983),False
6,25,EASE,7,593,"Silence of the Lambs, The (1991)",False
7,25,EASE,8,2959,Fight Club (1999),False
8,25,EASE,9,356,Forrest Gump (1994),False
9,25,EASE,10,91529,"Dark Knight Rises, The (2012)",True


## Interpretation

Compare `LLM shared shortlist` against **both** shortlist baselines. Compare it with the full-catalog
baselines to understand the entire pipeline. `LLM with EASE failure fallback` is a separate policy,
not a repaired pure-LLM score. A confidence interval spanning zero does not establish a gain.

Pilot metrics describe these 100 users; compare every model on those users rather than comparing
the pilot directly with the 594-user averages in earlier notebooks. All request/response caches and
data exports live under the ignored split-specific `notebook11_llm_reranking/` directory.